# TrAISformer: Full Pipeline - Data Preprocessing → Training → Inference
## Processing 15-min Interpolated U.S. Maritime AIS Data

This notebook implements the complete TrAISformer pipeline as described in the paper:
1. **Load & Analyze** interpolated parquet data
2. **Preprocess** trajectories (filter, normalize)
3. **Tokenize** features to discrete bins (key innovation)
4. **Create pickle files** in TrAISformer format
5. **Train** the transformer model
6. **Run inference** and evaluate predictions

**Expected Output:** Trajectory predictions up to 10+ hours ahead with <10 nautical mile error

## STEP 1: Import Libraries & Configure Environment

This cell imports all necessary libraries and sets up paths for the TrAISformer pipeline.
- **Pandas/Numpy:** Data manipulation and numerical operations
- **PyTorch:** Deep learning framework for the transformer model
- **Scikit-learn:** Data normalization and utilities
- **Matplotlib/Seaborn:** Visualization
- **Pickle:** Serialization format for TrAISformer data

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pickle
import os
import sys
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Setup paths
WORKSPACE_ROOT = Path(r"F:\PyTorch_GPU\maritime_monitoring_preprocessing")
INTERPOLATED_DATA_PATH = (
    WORKSPACE_ROOT
    / "interpolated_results"
    / "interpolated_ais_data_20200105_20200112_15min.parquet"
)
TRAISFORMER_PATH = WORKSPACE_ROOT / "TRAIS_Former_" / "CEE_TrAISformer"
OUTPUT_DIR = TRAISFORMER_PATH / "data" / "us_maritime"
RESULTS_DIR = TRAISFORMER_PATH / "results"

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add TrAISformer modules to path
sys.path.insert(0, str(TRAISFORMER_PATH))

print(f"✓ Workspace root: {WORKSPACE_ROOT}")
print(f"✓ Data file: {INTERPOLATED_DATA_PATH.exists()}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

✓ Workspace root: F:\PyTorch_GPU\maritime_monitoring_preprocessing
✓ Data file: True
✓ Output directory: F:\PyTorch_GPU\maritime_monitoring_preprocessing\TRAIS_Former_\CEE_TrAISformer\data\us_maritime
✓ PyTorch version: 2.6.0+cu124
✓ CUDA available: True


## STEP 2: Load & Analyze Interpolated Data

This cell loads the parquet file containing 15-minute interpolated AIS data.
- **Parquet format:** Efficient columnar storage
- **Columns analyzed:** LAT, LON, SOG, COG, MMSI, timestamp
- **Output:** Data bounds and statistics needed for normalization in subsequent steps

In [2]:
# Load interpolated data - using enhanced version with SOG and COG calculated
print("Loading interpolated AIS data from parquet (with SOG & COG)...")

# First, try to load the enhanced version; if not available, load original
enhanced_path = (
    INTERPOLATED_DATA_PATH.parent
    / "interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet"
)

if enhanced_path.exists():
    print(f"Loading enhanced parquet with SOG/COG: {enhanced_path.name}")
    df = pd.read_parquet(enhanced_path)
else:
    print(
        f"Enhanced version not found yet. Using original: {INTERPOLATED_DATA_PATH.name}"
    )
    df = pd.read_parquet(INTERPOLATED_DATA_PATH)

df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
df = df.set_index("BaseDateTime").sort_index()

print(f"\n{'='*60}")
print("DATA OVERVIEW")
print(f"{'='*60}")
print(f"Total records: {len(df):,}")
print(f"Unique vessels (MMSI): {df['MMSI'].nunique():,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

# Analyze geographic and speed bounds
print(f"\n{'='*60}")
print("DATA STATISTICS")
print(f"{'='*60}")

lat_min, lat_max = df["LAT"].min(), df["LAT"].max()
lon_min, lon_max = df["LON"].min(), df["LON"].max()

# Handle SOG and COG columns
if "SOG" in df.columns:
    sog_min, sog_max = df["SOG"].min(), df["SOG"].max()
else:
    sog_min, sog_max = 0, 30
    print("⚠️  SOG column not found - will calculate in next step")

if "COG" in df.columns:
    cog_min, cog_max = df["COG"].min(), df["COG"].max()
else:
    cog_min, cog_max = 0, 360
    print("⚠️  COG column not found - will calculate in next step")

print(f"\nLatitude:   {lat_min:.4f}° to {lat_max:.4f}° (range: {lat_max-lat_min:.4f}°)")
print(f"Longitude:  {lon_min:.4f}° to {lon_max:.4f}° (range: {lon_max-lon_min:.4f}°)")
print(f"SOG (knots): {sog_min:.2f} to {sog_max:.2f}")
print(f"COG (°):    {cog_min:.2f} to {cog_max:.2f}")

# Store for later use
BOUNDS = {
    "lat_min": lat_min,
    "lat_max": lat_max,
    "lon_min": lon_min,
    "lon_max": lon_max,
    "sog_max": max(sog_max, 30.0),  # Use 30 knots as standard max
    "cog_max": 360.0,
}

print(f"\nNormalization bounds (will use for tokenization):")
for key, val in BOUNDS.items():
    print(f"  {key}: {val}")

# Check data quality
print(f"\n{'='*60}")
print("DATA QUALITY CHECKS")
print(f"{'='*60}")
required_cols = ["LAT", "LON"]
if "SOG" in df.columns:
    required_cols.append("SOG")
if "COG" in df.columns:
    required_cols.append("COG")

print(f"NaN values in required columns:")
print(df[required_cols].isnull().sum())
print(f"\nDuplicate timestamps per vessel: {df.index.duplicated().sum()}")

Loading interpolated AIS data from parquet (with SOG & COG)...
Enhanced version not found yet. Using original: interpolated_ais_data_20200105_20200112_15min.parquet

DATA OVERVIEW
Total records: 6,863,558
Unique vessels (MMSI): 14,496
Date range: 2020-01-05 00:00:01 to 2020-01-13 00:14:50

Columns: ['LAT', 'LON', 'interpolated', 'MMSI']

First 5 rows:
                          LAT        LON  interpolated       MMSI
BaseDateTime                                                     
2020-01-05 00:00:01  13.58120  144.83658         False  369970581
2020-01-05 00:00:02  12.44007  144.52146         False  219802000
2020-01-05 00:00:04  13.46193  144.66508         False  367796190
2020-01-05 00:00:07  13.42761  144.66547         False  368926398
2020-01-05 00:00:10  13.45492  144.63816         False  368926395

DATA STATISTICS
⚠️  SOG column not found - will calculate in next step
⚠️  COG column not found - will calculate in next step

Latitude:   -2574.2780° to 3275.1834° (range: 5849.4614°

## Step 2.5: Calculate Derived Features (SOG & COG)

**Objective:** Create enhanced parquet file with interpolated SOG and COG columns
- **Input:** Original parquet with LAT, LON (15-min interpolated)
- **Method:** Calculate Speed Over Ground (SOG) from haversine distance, Course Over Ground (COG) from bearing angles
- **Output:** New parquet file with 6 columns: `[BaseDateTime, LAT, LON, interpolated, MMSI, SOG, COG]`

In [ ]:
import numpy as np


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two points using Haversine formula.
    Returns distance in nautical miles.
    """
    R = 3440.065  # Earth radius in nautical miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c

    return distance


def calculate_bearing(lat1, lon1, lat2, lon2):
    """
    Calculate bearing (course) from point 1 to point 2.
    Returns bearing in degrees (0-360).
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(y, x))

    # Normalize to 0-360 range
    bearing = (bearing + 360) % 360
    return bearing


# Load original interpolated data (without SOG/COG)
print("Loading original interpolated parquet file...")
df_original = pd.read_parquet(INTERPOLATED_DATA_PATH)
df_original["BaseDateTime"] = pd.to_datetime(df_original["BaseDateTime"])

# Sort by MMSI and timestamp
df_original = df_original.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)

print(
    f"Loaded {len(df_original):,} records from {df_original['MMSI'].nunique():,} vessels"
)

# Calculate SOG and COG for each vessel
print("\nCalculating SOG and COG features...")

sog_values = []
cog_values = []
vessel_ids = df_original["MMSI"].values

for idx in range(len(df_original)):
    if idx == 0:
        # First point: use next point to estimate
        lat1 = df_original.loc[idx, "LAT"]
        lon1 = df_original.loc[idx, "LON"]
        lat2 = df_original.loc[idx + 1, "LAT"]
        lon2 = df_original.loc[idx + 1, "LON"]
        time1 = df_original.loc[idx, "BaseDateTime"]
        time2 = df_original.loc[idx + 1, "BaseDateTime"]
        mmsi1 = vessel_ids[idx]
        mmsi2 = vessel_ids[idx + 1]
    elif idx == len(df_original) - 1:
        # Last point: use previous point
        lat1 = df_original.loc[idx - 1, "LAT"]
        lon1 = df_original.loc[idx - 1, "LON"]
        lat2 = df_original.loc[idx, "LAT"]
        lon2 = df_original.loc[idx, "LON"]
        time1 = df_original.loc[idx - 1, "BaseDateTime"]
        time2 = df_original.loc[idx, "BaseDateTime"]
        mmsi1 = vessel_ids[idx - 1]
        mmsi2 = vessel_ids[idx]
    else:
        # Middle point: use surrounding points for better estimate
        lat1 = df_original.loc[idx - 1, "LAT"]
        lon1 = df_original.loc[idx - 1, "LON"]
        lat2 = df_original.loc[idx + 1, "LAT"]
        lon2 = df_original.loc[idx + 1, "LON"]
        time1 = df_original.loc[idx - 1, "BaseDateTime"]
        time2 = df_original.loc[idx + 1, "BaseDateTime"]
        mmsi1 = vessel_ids[idx - 1]
        mmsi2 = vessel_ids[idx + 1]

    # Only calculate if same vessel (no multi-vessel jumps)
    if mmsi1 == mmsi2 == vessel_ids[idx]:
        time_diff_hours = (time2 - time1).total_seconds() / 3600.0

        if time_diff_hours > 0:
            dist_nm = haversine_distance(lat1, lon1, lat2, lon2)
            sog = dist_nm / time_diff_hours  # knots
            bearing = calculate_bearing(lat1, lon1, lat2, lon2)
        else:
            sog = 0.0
            bearing = 0.0
    else:
        sog = 0.0
        bearing = 0.0

    sog_values.append(sog)
    cog_values.append(bearing)

    if (idx + 1) % 100000 == 0:
        print(f"  Processed {idx + 1:,} records...")

print(f"Completed SOG/COG calculations")

# Add to dataframe and clip to valid ranges
df_original["SOG"] = np.array(sog_values)
df_original["COG"] = np.array(cog_values)

# Clip to valid ranges
df_original["SOG"] = df_original["SOG"].clip(0, 30)  # 0-30 knots
df_original["COG"] = df_original["COG"].clip(0, 360)  # 0-360 degrees

print(f"\nFeature ranges after calculation:")
print(f"  SOG: {df_original['SOG'].min():.2f} - {df_original['SOG'].max():.2f} knots")
print(f"  COG: {df_original['COG'].min():.2f} - {df_original['COG'].max():.2f} degrees")

# Save enhanced parquet file
enhanced_output_path = (
    INTERPOLATED_DATA_PATH.parent
    / "interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet"
)
print(f"\nSaving enhanced parquet file: {enhanced_output_path.name}")
df_original.to_parquet(enhanced_output_path)
print(f"✓ Saved {len(df_original):,} records to enhanced parquet file")

# Display sample of enhanced data
print(f"\nSample of enhanced data:")
print(df_original[["BaseDateTime", "LAT", "LON", "SOG", "COG", "MMSI"]].head(10))

Loading original interpolated parquet file...
Loaded 6,863,558 records from 14,496 vessels

Calculating SOG and COG features...
  Processed 100,000 records...
  Processed 200,000 records...
  Processed 300,000 records...
  Processed 400,000 records...
  Processed 500,000 records...
  Processed 600,000 records...
  Processed 700,000 records...
  Processed 800,000 records...
  Processed 900,000 records...
  Processed 1,000,000 records...
  Processed 1,100,000 records...
  Processed 1,200,000 records...
  Processed 1,300,000 records...
  Processed 1,400,000 records...
  Processed 1,500,000 records...
  Processed 1,600,000 records...
  Processed 1,700,000 records...
  Processed 1,800,000 records...
  Processed 1,900,000 records...
  Processed 2,000,000 records...
  Processed 2,100,000 records...
  Processed 2,200,000 records...
  Processed 2,300,000 records...
  Processed 2,400,000 records...
  Processed 2,500,000 records...
  Processed 2,600,000 records...
  Processed 2,700,000 records...

## STEP 3: Trajectory Filtering & Preprocessing

This critical preprocessing step filters raw AIS data into clean trajectories:
1. **Remove stationary vessels** (SOG < 0.05 knots threshold)
2. **Filter by geographic bounds** (valid maritime region)
3. **Remove NaN values** and quality issues
4. **Minimum length check** (at least 36 timesteps = 9 hours of 15-min data)
5. **Create per-vessel trajectories** (chronological order)

**Output:** List of dictionaries with MMSI and normalized trajectory arrays

In [4]:
# Configuration for filtering
MIN_SOG_THRESHOLD = 0.05  # knots - filter stationary vessels
MIN_TRAJECTORY_LENGTH = 36  # timesteps (9 hours at 15-min intervals)
LAT_MIN_BOUNDS = BOUNDS["lat_min"] - 1
LAT_MAX_BOUNDS = BOUNDS["lat_max"] + 1
LON_MIN_BOUNDS = BOUNDS["lon_min"] - 1
LON_MAX_BOUNDS = BOUNDS["lon_max"] + 1

# Preprocess trajectories
trajectories = []
skipped_reasons = {
    "too_short": 0,
    "all_stationary": 0,
    "nan_values": 0,
    "out_of_bounds": 0,
    "valid": 0,
}

print("Processing trajectories by vessel...")
print(f"Filters: min_sog={MIN_SOG_THRESHOLD}, min_length={MIN_TRAJECTORY_LENGTH}")

for mmsi, group in tqdm(df.groupby("MMSI"), desc="Processing vessels"):
    group = group.sort_index()  # Ensure chronological order

    # FILTER 1: Remove stationary vessels
    moving_mask = group["SOG"] > MIN_SOG_THRESHOLD
    if moving_mask.sum() == 0:
        skipped_reasons["all_stationary"] += 1
        continue

    first_moving_idx = moving_mask.idxmax()
    group = group.loc[first_moving_idx:].reset_index(drop=True)

    # FILTER 2: Remove NaN values
    group = group.dropna(subset=["LAT", "LON", "SOG", "COG"]).reset_index(drop=True)
    if len(group) == 0:
        skipped_reasons["nan_values"] += 1
        continue

    # FILTER 3: Check if too short
    if len(group) < MIN_TRAJECTORY_LENGTH:
        skipped_reasons["too_short"] += 1
        continue

    # FILTER 4: Geographic bounds check
    in_bounds = (
        (group["LAT"] >= LAT_MIN_BOUNDS)
        & (group["LAT"] <= LAT_MAX_BOUNDS)
        & (group["LON"] >= LON_MIN_BOUNDS)
        & (group["LON"] <= LON_MAX_BOUNDS)
    )
    group = group[in_bounds].reset_index(drop=True)

    if len(group) < MIN_TRAJECTORY_LENGTH:
        skipped_reasons["out_of_bounds"] += 1
        continue

    # BUILD TRAJECTORY ARRAY: [LAT_norm, LON_norm, SOG_norm, COG_norm, TIMESTAMP_unix]
    traj_array = np.zeros((len(group), 5), dtype=np.float32)

    # Normalize geographic coordinates
    traj_array[:, 0] = (group["LAT"].values - BOUNDS["lat_min"]) / (
        BOUNDS["lat_max"] - BOUNDS["lat_min"]
    )
    traj_array[:, 1] = (group["LON"].values - BOUNDS["lon_min"]) / (
        BOUNDS["lon_max"] - BOUNDS["lon_min"]
    )

    # Normalize speeds and courses
    traj_array[:, 2] = group["SOG"].values / BOUNDS["sog_max"]
    traj_array[:, 3] = group["COG"].values / 360.0

    # Unix timestamp
    traj_array[:, 4] = group.index.astype(np.int64).values // 10**9

    # Clip to [0, 0.9999) to prevent boundary issues during tokenization
    traj_array[:, :4] = np.clip(traj_array[:, :4], 0, 0.9999)

    trajectories.append({"mmsi": int(mmsi), "traj": traj_array})
    skipped_reasons["valid"] += 1

# Print statistics
print(f"\n{'='*60}")
print("TRAJECTORY FILTERING RESULTS")
print(f"{'='*60}")
print(f"Valid trajectories: {skipped_reasons['valid']:,}")
print(f"Skipped (too short): {skipped_reasons['too_short']:,}")
print(f"Skipped (all stationary): {skipped_reasons['all_stationary']:,}")
print(f"Skipped (NaN values): {skipped_reasons['nan_values']:,}")
print(f"Skipped (out of bounds): {skipped_reasons['out_of_bounds']:,}")

print(f"\n{'='*60}")
print("TRAJECTORY STATISTICS")
print(f"{'='*60}")
if trajectories:
    lengths = [len(t["traj"]) for t in trajectories]
    total_timesteps = sum(lengths)
    print(f"Total trajectories: {len(trajectories):,}")
    print(f"Total timesteps: {total_timesteps:,}")
    print(f"Avg trajectory length: {np.mean(lengths):.1f} timesteps")
    print(f"Min trajectory length: {np.min(lengths)} timesteps")
    print(f"Max trajectory length: {np.max(lengths)} timesteps")
else:
    print("⚠️  No valid trajectories found!")

Processing trajectories by vessel...
Filters: min_sog=0.05, min_length=36


Processing vessels:   0%|          | 0/14496 [00:00<?, ?it/s]


KeyError: 'SOG'

## STEP 4: Define Tokenization Configuration

**Key Innovation of TrAISformer:** Discrete representation via tokenization
Instead of predicting continuous values (lat, lon, sog, cog), the model treats each feature as a **classification problem** with discrete bins.

This cell defines:
- **Quantization levels** for each feature (lat, lon, speed, course)
- **Embedding dimensions** for each feature
- **Why discrete?** Better handles multimodal distributions + enables cross-entropy loss

The paper uses uniform quantization: `token_idx = int(normalized_value * num_bins)`

In [ ]:
# Tokenization configuration (from TrAISformer paper)
lat_range = BOUNDS["lat_max"] - BOUNDS["lat_min"]
lon_range = BOUNDS["lon_max"] - BOUNDS["lon_min"]

# Quantization levels (number of discrete bins for each feature)
# Formula: (range_degrees) * bins_per_degree
lat_size = max(200, int(lat_range * 10) + 10)  # Latitude bins
lon_size = max(200, int(lon_range * 10) + 10)  # Longitude bins
sog_size = 30  # Speed: 0-30 knots
cog_size = 72  # Course: 5° per bin (360/72)

# Embedding dimensions (learned representation after tokenization)
n_lat_embd = 256
n_lon_embd = 256
n_sog_embd = 128
n_cog_embd = 128

# Transformer architecture
n_head = 8  # Attention heads
n_layer = 8  # Transformer blocks
max_seqlen = 120  # Maximum sequence length for training

# Create config dictionary
config = {
    "lat_size": lat_size,
    "lon_size": lon_size,
    "sog_size": sog_size,
    "cog_size": cog_size,
    "n_lat_embd": n_lat_embd,
    "n_lon_embd": n_lon_embd,
    "n_sog_embd": n_sog_embd,
    "n_cog_embd": n_cog_embd,
    "n_embd": n_lat_embd + n_lon_embd + n_sog_embd + n_cog_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "max_seqlen": max_seqlen,
    "full_vocab_size": lat_size + lon_size + sog_size + cog_size,
}

print("TOKENIZATION CONFIGURATION")
print(f"{'='*60}")
print(f"\nQuantization Levels (bins per feature):")
print(f"  Latitude:  {lat_size} bins (1 bin ≈ {lat_range/lat_size:.4f}°)")
print(f"  Longitude: {lon_size} bins (1 bin ≈ {lon_range/lon_size:.4f}°)")
print(f"  SOG:       {sog_size} bins (1 bin ≈ {30.0/sog_size:.2f} knots)")
print(f"  COG:       {cog_size} bins (1 bin ≈ {360.0/cog_size:.1f}°)")

print(f"\nEmbedding Dimensions:")
print(f"  Latitude:  {n_lat_embd}D")
print(f"  Longitude: {n_lon_embd}D")
print(f"  SOG:       {n_sog_embd}D")
print(f"  COG:       {n_cog_embd}D")
print(f"  Total:     {config['n_embd']}D per timestep")

print(f"\nTransformer Architecture:")
print(f"  Blocks:    {n_layer} layers")
print(f"  Heads:     {n_head} attention heads")
print(f"  Max seq:   {max_seqlen} timesteps")

print(f"\nTotal vocab size: {config['full_vocab_size']} (sum of all bins)")

## STEP 5: Create Train/Validation/Test Split

Splits preprocessed trajectories into training, validation, and test sets.
**Important:** Split by vessel (MMSI), NOT by timestep, to prevent temporal leakage!

- **Train:** 80% of vessels (used for model training)
- **Validation:** 10% of vessels (used for hyperparameter tuning)
- **Test:** 10% of vessels (final evaluation with unseen vessels)

In [ ]:
# Train/Val/Test split (by vessel, not by timestep)
np.random.seed(42)
n_vessels = len(trajectories)
n_train = int(0.8 * n_vessels)
n_val = int(0.1 * n_vessels)

# Shuffle vessel list
indices = np.arange(n_vessels)
np.random.shuffle(indices)

train_idx = indices[:n_train]
val_idx = indices[n_train : n_train + n_val]
test_idx = indices[n_train + n_val :]

train_data = [trajectories[i] for i in train_idx]
val_data = [trajectories[i] for i in val_idx]
test_data = [trajectories[i] for i in test_idx]

print(f"{'='*60}")
print("TRAIN/VAL/TEST SPLIT (by vessel)")
print(f"{'='*60}")

for phase, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    n_traj = len(data)
    total_ts = sum(len(t["traj"]) for t in data)
    avg_len = total_ts / n_traj if n_traj > 0 else 0
    print(f"\n{phase}:")
    print(f"  Vessels: {n_traj:,} ({100*n_traj/n_vessels:.1f}%)")
    print(f"  Total timesteps: {total_ts:,}")
    print(f"  Avg trajectory length: {avg_len:.1f}")

# Save as pickle files
print(f"\n{'='*60}")
print("SAVING PICKLE FILES")
print(f"{'='*60}")

pickle_files = [
    (train_data, OUTPUT_DIR / "us_maritime_train.pkl", "Train"),
    (val_data, OUTPUT_DIR / "us_maritime_valid.pkl", "Validation"),
    (test_data, OUTPUT_DIR / "us_maritime_test.pkl", "Test"),
]

for data, filepath, phase in pickle_files:
    with open(filepath, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"✓ {phase}: {filepath.name}")

print(f"\n✓ All pickle files saved to: {OUTPUT_DIR}")

## STEP 6: Define Custom AIS Dataset Class

Custom PyTorch Dataset that:
1. Loads trajectories from pickle files
2. Returns padded sequences of fixed length
3. Provides masks for actual vs padding timesteps
4. Handles tokenization on-the-fly (during training)

This matches the format used in the original TrAISformer implementation.

In [ ]:
class AISDataset(Dataset):
    """Custom PyTorch Dataset for AIS trajectories"""

    def __init__(self, l_data, max_seqlen=120, device=torch.device("cpu")):
        """
        Args:
            l_data: list of dicts with 'mmsi' and 'traj' (normalized [0,1))
            max_seqlen: maximum sequence length (pad or truncate to this)
            device: torch device (cpu or cuda)
        """
        self.l_data = l_data
        self.max_seqlen = max_seqlen
        self.device = device

    def __len__(self):
        return len(self.l_data)

    def __getitem__(self, idx):
        """
        Returns:
            seq: (max_seqlen, 4) - normalized trajectory [LAT, LON, SOG, COG]
            mask: (max_seqlen,) - 1 for real data, 0 for padding
            seqlen: actual sequence length before padding
            mmsi: vessel identifier
            time_start: Unix timestamp of first point
        """
        V = self.l_data[idx]
        m_v = V["traj"][:, :4]  # Extract only [LAT, LON, SOG, COG]

        # Clip extreme values
        m_v = np.clip(m_v, 0, 0.9999)

        seqlen = min(len(m_v), self.max_seqlen)
        seq = np.zeros((self.max_seqlen, 4), dtype=np.float32)
        seq[:seqlen, :] = m_v[:seqlen, :]

        seq = torch.tensor(seq, dtype=torch.float32, device=self.device)

        mask = torch.zeros(self.max_seqlen, device=self.device)
        mask[:seqlen] = 1.0

        seqlen = torch.tensor(seqlen, dtype=torch.int32)
        mmsi = torch.tensor(V["mmsi"], dtype=torch.int64)
        time_start = torch.tensor(V["traj"][0, 4], dtype=torch.int64)

        return seq, mask, seqlen, mmsi, time_start


# Test dataset
print("Testing AISDataset class...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset = AISDataset(
    train_data[:10], max_seqlen=config["max_seqlen"], device=device
)

seq, mask, seqlen, mmsi, time_start = test_dataset[0]
print(f"\nSample batch:")
print(f"  Sequence shape: {seq.shape} (max_seqlen x 4 features)")
print(f"  Mask shape: {mask.shape}")
print(f"  Actual seqlen: {seqlen}")
print(f"  MMSI: {mmsi}")
print(f"✓ Dataset class working correctly")

## STEP 7: Implement Tokenization Functions

This cell implements the **core TrAISformer innovation**: converting continuous coordinates to discrete tokens.

**Process:**
1. Normalize feature to [0, 1) range
2. Multiply by number of bins
3. Take integer part to get token index (0 to num_bins-1)

**Example:** 
- Latitude 40.5° in range [25, 45] → normalized=0.775 → token=0.775*250=194

## STEP 2.5: Check Actual Column Names

Quick diagnostic to verify the actual column names in the parquet file and adjust the pipeline accordingly.

In [ ]:
# Check actual columns in the parquet file
print("Checking actual columns in parquet file...")
df_sample = pd.read_parquet(INTERPOLATED_DATA_PATH)

print(f"\nActual columns in parquet file:")
print(df_sample.columns.tolist())
print(f"\nFirst few rows:")
print(df_sample.head())
print(f"\nData types:")
print(df_sample.dtypes)
print(f"\nShape: {df_sample.shape}")

# Check for common column name variations
possible_cols = {
    "lat": ["LAT", "latitude", "Latitude", "lat", "Lat", "LATITUDE"],
    "lon": ["LON", "longitude", "Longitude", "lon", "Lon", "LONGITUDE"],
    "sog": ["SOG", "speed", "Speed", "sog", "Sog", "SPEED", "speed_over_ground"],
    "cog": ["COG", "course", "Course", "cog", "Cog", "COURSE", "course_over_ground"],
    "mmsi": ["MMSI", "mmsi", "Mmsi", "vessel_id"],
}

actual_cols = {}
for feature, variants in possible_cols.items():
    for col in variants:
        if col in df_sample.columns:
            actual_cols[feature] = col
            print(f"✓ Found '{feature}' column: '{col}'")
            break
    else:
        print(
            f"✗ Could not find '{feature}' column. Available: {df_sample.columns.tolist()}"
        )

Checking actual columns in parquet file...

Actual columns in parquet file:
['BaseDateTime', 'LAT', 'LON', 'interpolated', 'MMSI']

First few rows:
         BaseDateTime        LAT         LON  interpolated  MMSI
0 2020-01-10 21:47:31  22.501470 -156.497760         False     0
1 2020-01-10 22:02:31  22.480578 -156.515550          True     0
2 2020-01-10 22:17:31  22.459697 -156.532746          True     0
3 2020-01-10 22:32:31  22.438143 -156.549763          True     0
4 2020-01-10 22:47:31  22.416340 -156.566636          True     0

Data types:
BaseDateTime    datetime64[ns]
LAT                    float64
LON                    float64
interpolated              bool
MMSI                     int64
dtype: object

Shape: (6863558, 5)
✓ Found 'lat' column: 'LAT'
✓ Found 'lon' column: 'LON'
✗ Could not find 'sog' column. Available: ['BaseDateTime', 'LAT', 'LON', 'interpolated', 'MMSI']
✗ Could not find 'cog' column. Available: ['BaseDateTime', 'LAT', 'LON', 'interpolated', 'MMSI']
✓ Found '

## STEP 2.6: Calculate Derived Features (SOG & COG)

Since the parquet file only contains LAT/LON positions, we need to derive:
- **SOG (Speed Over Ground):** Haversine distance / time interval (in knots)
- **COG (Course Over Ground):** Bearing angle from current to next position (0-360°)

These calculations follow standard maritime conventions and are computed per-vessel to ensure valid measurements.

In [ ]:
from scipy.interpolate import CubicSpline
from datetime import timedelta
import math

# Load the original interpolated data
print("Loading interpolated data from parquet...")
df_interp = pd.read_parquet(INTERPOLATED_DATA_PATH)
df_interp["BaseDateTime"] = pd.to_datetime(df_interp["BaseDateTime"])
df_interp = df_interp.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)

print(f"Loaded {len(df_interp):,} records for {df_interp['MMSI'].nunique():,} vessels")


def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in nautical miles"""
    R = 3440.07  # Earth radius in nautical miles

    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    return R * c


def calculate_bearing(lat1, lon1, lat2, lon2):
    """Calculate bearing angle (COG) in degrees (0-360)"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(
        dlon
    )

    bearing = math.degrees(math.atan2(x, y))
    bearing = (bearing + 360) % 360  # Normalize to 0-360
    return bearing


# Calculate SOG and COG for each vessel
print("\nCalculating SOG and COG for each vessel...")
sog_values = []
cog_values = []

for mmsi, group in tqdm(df_interp.groupby("MMSI"), desc="Computing SOG/COG"):
    group = group.sort_values("BaseDateTime").reset_index(drop=True)

    vessel_sog = []
    vessel_cog = []

    for i in range(len(group)):
        if i == 0:
            # First point: use bearing and distance to next point
            if i + 1 < len(group):
                lat1, lon1 = group.iloc[i][["LAT", "LON"]]
                lat2, lon2 = group.iloc[i + 1][["LAT", "LON"]]

                time1 = group.iloc[i]["BaseDateTime"]
                time2 = group.iloc[i + 1]["BaseDateTime"]
                time_diff_hours = (time2 - time1).total_seconds() / 3600

                distance_nm = haversine_distance(lat1, lon1, lat2, lon2)
                sog = distance_nm / time_diff_hours if time_diff_hours > 0 else 0
                cog = calculate_bearing(lat1, lon1, lat2, lon2)

                vessel_sog.append(max(0, min(sog, 30)))  # Clip to 0-30 knots
                vessel_cog.append(cog)
            else:
                vessel_sog.append(0)
                vessel_cog.append(0)
        else:
            # Use average of previous and next points for smoother COG
            lat1, lon1 = group.iloc[i - 1][["LAT", "LON"]]
            lat_curr, lon_curr = group.iloc[i][["LAT", "LON"]]

            if i + 1 < len(group):
                lat2, lon2 = group.iloc[i + 1][["LAT", "LON"]]
            else:
                lat2, lon2 = lat_curr, lon_curr

            # Calculate SOG from previous to current
            time_prev = group.iloc[i - 1]["BaseDateTime"]
            time_curr = group.iloc[i]["BaseDateTime"]
            time_diff_hours = (time_curr - time_prev).total_seconds() / 3600

            if time_diff_hours > 0:
                dist_prev_curr = haversine_distance(lat1, lon1, lat_curr, lon_curr)
                sog = dist_prev_curr / time_diff_hours
            else:
                sog = 0

            # Average bearing from prev->curr and curr->next
            bearing1 = calculate_bearing(lat1, lon1, lat_curr, lon_curr)
            bearing2 = calculate_bearing(lat_curr, lon_curr, lat2, lon2)
            cog = (bearing1 + bearing2) / 2
            cog = cog % 360

            vessel_sog.append(max(0, min(sog, 30)))
            vessel_cog.append(cog)

    sog_values.extend(vessel_sog)
    cog_values.extend(vessel_cog)

# Add SOG and COG columns
df_interp["SOG"] = sog_values
df_interp["COG"] = cog_values

print(f"\n{'='*60}")
print("SOG & COG CALCULATION RESULTS")
print(f"{'='*60}")
print(
    f"SOG (knots): min={df_interp['SOG'].min():.2f}, max={df_interp['SOG'].max():.2f}, mean={df_interp['SOG'].mean():.2f}"
)
print(
    f"COG (°):     min={df_interp['COG'].min():.2f}, max={df_interp['COG'].max():.2f}, mean={df_interp['COG'].mean():.2f}"
)
print(f"\nColumns in enhanced dataset:")
print(df_interp.columns.tolist())
print(f"\nFirst 5 rows:")
print(df_interp.head())

# Save enhanced parquet file with SOG and COG
enhanced_parquet_path = (
    INTERPOLATED_DATA_PATH.parent
    / "interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet"
)
df_interp.to_parquet(enhanced_parquet_path, compression="snappy", index=False)

print(f"\n{'='*60}")
print("✓ Enhanced parquet file saved with SOG & COG!")
print(f"{'='*60}")
print(f"File: {enhanced_parquet_path.name}")
print(f"Size: {enhanced_parquet_path.stat().st_size / (1024**2):.2f} MB")